# 1. Install Dependencies"

In [ ]:
# Pinned versions tested on Kaggle T4 GPU
# Kaggle pre-installs torch with CUDA; only install missing packages
!pip install --quiet \
    transformers==4.44.3 \
    peft==0.12.0 \
    trl==0.10.0 \
    datasets==2.21.0 \
    bitsandbytes==0.43.3 \
    accelerate==0.34.2 \
    huggingface-hub==0.27.0 \
    pyarrow==16.0.0 \
    tqdm==4.66.5

# 2. HuggingFace Hub Authentication

In [ ]:
import os
from huggingface_hub import login, whoami, HfApi

# Read HF_TOKEN from environment (Kaggle secrets or .env)
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    # In Kaggle, use the secrets mechanism
    from kaggle_secrets import SecretBucket
    bucket = SecretBucket()
    HF_TOKEN = bucket.check_secret("HF_TOKEN")

print(f"HF_TOKEN: {'***' if HF_TOKEN else 'NOT SET'}")

# Login and verify
if HF_TOKEN:
    login(token=HF_TOKEN)
    user = whoami()
    print(f"Logged in as: {user['name']}")
else:
    raise ValueError("HF_TOKEN not set. Set in Kaggle secrets or environment.")

# 3. Load Training Data

In [ ]:
import json
from pathlib import Path

# Kaggle notebook input path
KAGGLE_INPUT_PATH = "/kaggle/input/paperlens-extraction"

# Check if using Kaggle input, otherwise use local path
if Path(KAGGLE_INPUT_PATH).exists():
    data_path = Path(KAGGLE_INPUT_PATH)
    print(f"Using Kaggle input path: {data_path}")
else:
    data_path = Path("./data/extraction_dataset")
    print(f"Using local path: {data_path}")

train_path = data_path / "train.jsonl"
val_path = data_path / "val.jsonl"
test_path = data_path / "test.jsonl" if (data_path / "test.jsonl").exists() else None

print(f"Train: {train_path}")
print(f"Val: {val_path}")

# Load data
def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl(train_path)
val_data = load_jsonl(val_path)

print(f"Loaded {len(train_data)} training examples")
print(f"Loaded {len(val_data)} validation examples")

# Show sample
print("\nSample training example:")
print(json.dumps(train_data[0]['messages'][1], indent=2)[:500], "...")

# 4. Load Base Model (4-bit QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# 4-bit NF4 quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} with 4-bit NF4 quantization...")
print("Expected VRAM: ~4-5 GB for quantized base model")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

# Verify GPU memory usage
import gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory after model load: allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")
else:
    print("Warning: CUDA not available")

model.gradient_checkpointing_enable()
model.config.use_cache = False

print("Model loaded successfully!")

# 5. Configure LoRA Adapter

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print("LoRA Configuration:")
print(f"  r (rank): {lora_config.r}")
print(f"  alpha: {lora_config.lora_alpha}")
print(f"  dropout: {lora_config.lora_dropout}")
print(f"  target_modules: {lora_config.target_modules}")
print(f"  bias: {lora_config.bias}")

model = get_peft_model(model, lora_config)
print(f"LoRA adapter configured. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# 6. Configure SFTTrainer

In [ ]:
from trl import SFTTrainer, DataCollatorForCompletionOnly
from transformers import TrainingArguments

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "./paperlens-extraction-adapter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="paged_adamw_32bit",
    max_seq_length=1024,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

print("Training Arguments:")
print(f"  per_device_train_batch_size: {training_args.per_device_train_batch_size}")
print(f"  gradient_accumulation_steps: {training_args.gradient_accumulation_steps}")
print(f"  effective_batch_size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  num_train_epochs: {training_args.num_train_epochs}")
print(f"  learning_rate: {training_args.learning_rate}")
print(f"  max_seq_length: {training_args.max_seq_length}")
print(f"  optim: {training_args.optim}")

# Create data collator
collator = DataCollatorForCompletionOnly(
    response_template="\\n\\n",
    tokenizer=tokenizer,
    mlm=False,
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    data_collator=collator,
    dataset_text_field="messages",
)

# 7. Training

In [ ]:
import math

print("Starting training...")
print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Expected training time: ~2-3 hours on Kaggle T4")

trainer.train()

# Log training loss per epoch
print("\nTraining completed!")
print(f"Total training steps: {trainer.state.global_step}")

# Get final metrics
metrics = trainer.log_metrics("eval", trainer.state.log_history[-1] if trainer.state.log_history else {})
print(f"Final eval loss: {metrics.get('eval_loss', 'N/A'):.4f}")

# 8. Save Adapter

In [ ]:
print(f"Saving adapter to {OUTPUT_DIR}/...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Verify saved files
import os
saved_files = os.listdir(OUTPUT_DIR)
print(f"Saved files: {saved_files}")

# Check adapter files
adapter_config = os.path.exists(f"{OUTPUT_DIR}/adapter_config.json")
adapter_model = os.path.exists(f"{OUTPUT_DIR}/adapter_model.bin")
print(f"adapter_config.json: {adapter_config}")
print(f"adapter_model.bin: {adapter_model}")

# 9. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi, create_repo, upload_folder
import shutil

HF_REPO_ID = os.environ.get("HF_HUB_MODEL_ID", "muhasin/paperlens-qwen2.5-3b-extraction")
HF_TOKEN = os.environ.get("HF_TOKEN")

print(f"Pushing to HuggingFace Hub: {HF_REPO_ID}")

# Create repo if it doesn't exist
api = HfApi()
try:
    api.create_repo(
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
        exist_ok=True,
        repo_type="model",
    )
    print(f"Repository {HF_REPO_ID} ready")
except Exception as e:
    print(f"Repo creation (may already exist): {e}")

# Upload adapter folder
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
    repo_type="model",
)
print("Adapter pushed to HuggingFace Hub!")

# Verify
files = api.list_repo_files(repo_id=HF_REPO_ID, token=HF_TOKEN, repo_type="model")
print(f"Hub files: {files}")

# 10. Smoke Test

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, Pipeline
import re
import json

print("Running smoke test on validation data...")

# Three test prompts as required:
# 1. A method-section chunk — expected output contains non-null "method" field
# 2. A results-section chunk — expected output contains non-null "key_finding" field  
# 3. A chunk from a different paper than training data — tests generalisation

test_prompts = [
    ("method", "Extract structured information from the following ML paper chunk:\n\nWe apply layer normalization to each layer..."),
    ("results", "Extract structured information from the following ML paper chunk:\n\nOur experiments demonstrate that..."),
    ("novel", "Extract structured information from the following ML paper chunk:\n\nThis is from a paper not in training data..."),
]

# Load base model and adapter
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Load adapter
adapter_model = PeftModel.from_pretrained(base_model, HF_REPO_ID)
adapter_model.eval()

results = []
for section, prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(adapter_model.device)
    
    with torch.no_grad():
        outputs = adapter_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract assistant response
    if "assistant" in response:
        response_text = response.split("assistant")[-1].strip()
    else:
        response_text = response
    
    # Try to parse as JSON
    try:
        # Find JSON object in response
        json_match = re.search(r'\{.*\}', response_text)
        if json_match:
            parsed = json.loads(json_match.group())
        else:
            parsed = {"raw": response_text[:200]}
        
        results.append({
            "section": section,
            "parsed": parsed,
            "valid_json": True
        })
        print(f"\nTest ({section}): Valid JSON")
        print(f"  Fields present: {list(parsed.keys())}")
        
        # For method section, check that method field exists
        if section == "method":
            has_method = "method" in parsed and parsed["method"] is not None
            print(f"  Has method field: {has_method}")
        
        # For results section, check that key_finding field exists  
        if section == "results":
            has_finding = "key_finding" in parsed and parsed["key_finding"] is not None
            print(f"  Has key_finding field: {has_finding}")
    except json.JSONDecodeError as e:
        results.append({
            "section": section,
            "error": str(e),
            "valid_json": False
        })
        print(f"\nTest ({section}): INVALID JSON - {e}")
        print(f"  Response preview: {response_text[:200]}...")

# Summary
print("\n" + "="*50)
print("Smoke Test Summary:")
print(f"  Total tests: {len(results)}")
print(f"  Valid JSON: {sum(1 for r in results if r.get('valid_json', False))}")
print(f"  All tests passed: {all(r.get('valid_json', False) for r in results)}")